In [3]:
"""
=============================================================================
COMPONENT I: RNN / LSTM Based Sequential Data Generation
=============================================================================
"""

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import warnings
warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────
# STEP 1: Load Dataset
# ─────────────────────────────────────────────
text_data = """
artificial intelligence systems learn patterns from data.
sequence models process information step by step.
recurrent neural networks are useful for sequence prediction.
lstm networks handle long term dependencies.
deep learning models improve sequence learning.
generative models create new samples from learned patterns.
language models predict the next word in a sentence.
sequence generation is used in chatbots and assistants.
machine learning helps computers learn automatically.
training data improves model accuracy.
neural networks simulate human brain structures.
optimization algorithms improve learning efficiency.
technology is transforming modern education.
online learning platforms use artificial intelligence.
students benefit from intelligent tutoring systems.
automation improves productivity and decision making.
""".strip()

print("=" * 60)
print("COMPONENT I: LSTM Based Sequential Data Generation")
print("=" * 60)
print(f"\n[Dataset] Total characters : {len(text_data)}")

# ─────────────────────────────────────────────
# STEP 2: Character-level Tokenization
# ─────────────────────────────────────────────
chars     = sorted(set(text_data))
char2idx  = {c: i for i, c in enumerate(chars)}
idx2char  = {i: c for c, i in char2idx.items()}
vocab_size = len(chars)
encoded   = [char2idx[c] for c in text_data]

print(f"[Dataset] Vocabulary size  : {vocab_size}")
print(f"[Dataset] Sample mapping   : 'a' -> {char2idx.get('a', '?')}, "
      f"'e' -> {char2idx.get('e', '?')}, ' ' -> {char2idx.get(' ', '?')}")

# ─────────────────────────────────────────────
# STEP 3: Create Input-Output Sequence Pairs
# ─────────────────────────────────────────────
SEQ_LEN    = 40
BATCH_SIZE = 64

class CharDataset(Dataset):
    def __init__(self, data, seq_len):
        self.data    = data
        self.seq_len = seq_len

    def __len__(self):
        return len(self.data) - self.seq_len

    def __getitem__(self, idx):
        x = torch.tensor(self.data[idx: idx + self.seq_len],     dtype=torch.long)
        y = torch.tensor(self.data[idx + 1: idx + self.seq_len + 1], dtype=torch.long)
        return x, y

dataset = CharDataset(encoded, SEQ_LEN)
loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

print(f"\n[Sequences] Sequence length : {SEQ_LEN} characters")
print(f"[Sequences] Total samples   : {len(dataset)}")
print(f"[Sequences] Batches/epoch   : {len(loader)}")

# ─────────────────────────────────────────────
# STEP 4: Design LSTM Generative Model
# ─────────────────────────────────────────────
class LSTMGenerator(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=256,
                 num_layers=2, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm      = nn.LSTM(embed_dim, hidden_dim, num_layers,
                                  batch_first=True, dropout=dropout)
        self.fc        = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden=None):
        emb         = self.embedding(x)
        out, hidden = self.lstm(emb, hidden)
        logits      = self.fc(out)
        return logits, hidden

device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
lstm_model = LSTMGenerator(vocab_size).to(device)
total_params = sum(p.numel() for p in lstm_model.parameters())

print(f"\n[Model] Architecture : Embedding({vocab_size}, 64) -> LSTM(64, 256, layers=2) -> Linear(256, {vocab_size})")
print(f"[Model] Total params : {total_params:,}")
print(f"[Model] Device       : {device}")

# ─────────────────────────────────────────────
# STEP 5: Train the Model
# ─────────────────────────────────────────────
EPOCHS = 100
LR     = 0.003

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(lstm_model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.5)

print(f"\n[Training] Epochs     : {EPOCHS}")
print(f"[Training] LR         : {LR}  (StepLR: step=30, gamma=0.5)")
print(f"[Training] Loss fn    : CrossEntropyLoss")
print(f"\n{'Epoch':>8}  {'Avg Loss':>10}  {'LR':>10}")
print("-" * 35)

for epoch in range(1, EPOCHS + 1):
    lstm_model.train()
    total_loss = 0.0

    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits, _ = lstm_model(xb)
        loss = criterion(logits.reshape(-1, vocab_size), yb.reshape(-1))
        loss.backward()
        nn.utils.clip_grad_norm_(lstm_model.parameters(), max_norm=5)
        optimizer.step()
        total_loss += loss.item()

    scheduler.step()

    if epoch % 20 == 0:
        avg_loss  = total_loss / len(loader)
        cur_lr    = scheduler.get_last_lr()[0]
        print(f"{epoch:>8}  {avg_loss:>10.4f}  {cur_lr:>10.6f}")

# ─────────────────────────────────────────────
# STEP 6: Generate New Sequences
# ─────────────────────────────────────────────
def generate_lstm(model, seed, length=200, temperature=0.8):
    """
    Generate text autoregressively from a seed string.
    temperature < 1 -> more focused, temperature > 1 -> more random.
    """
    model.eval()
    with torch.no_grad():
        chars_in      = [char2idx.get(c, 0) for c in seed]
        inp           = torch.tensor([chars_in], dtype=torch.long).to(device)
        _, hidden     = model(inp)
        generated     = seed

        for _ in range(length):
            last          = torch.tensor(
                [[char2idx.get(generated[-1], 0)]], dtype=torch.long
            ).to(device)
            logits, hidden = model(last, hidden)
            probs          = torch.softmax(
                logits[0, -1] / temperature, dim=-1
            ).cpu().numpy()
            next_c         = idx2char[np.random.choice(len(probs), p=probs)]
            generated     += next_c

    return generated


print("\n" + "=" * 60)
print("EXPECTED OUTPUT — GENERATED TEXT SEQUENCES (LSTM)")
print("=" * 60)

seeds       = ["deep learning", "language model", "neural network"]
temperatures = [0.6, 0.8, 1.0]

for seed in seeds:
    print(f"\n{'─'*55}")
    print(f"  Seed : \"{seed}\"")
    for temp in temperatures:
        out = generate_lstm(lstm_model, seed, length=120, temperature=temp)
        print(f"\n  [temperature={temp}]")
        print(f"  {out}")

COMPONENT I: LSTM Based Sequential Data Generation

[Dataset] Total characters : 832
[Dataset] Vocabulary size  : 28
[Dataset] Sample mapping   : 'a' -> 3, 'e' -> 7, ' ' -> 1

[Sequences] Sequence length : 40 characters
[Sequences] Total samples   : 792
[Sequences] Batches/epoch   : 13

[Model] Architecture : Embedding(28, 64) -> LSTM(64, 256, layers=2) -> Linear(256, 28)
[Model] Total params : 865,052
[Model] Device       : cuda

[Training] Epochs     : 100
[Training] LR         : 0.003  (StepLR: step=30, gamma=0.5)
[Training] Loss fn    : CrossEntropyLoss

   Epoch    Avg Loss          LR
-----------------------------------
      20      0.1187    0.003000
      40      0.1042    0.001500
      60      0.1031    0.000750
      80      0.0997    0.000750
     100      0.0988    0.000375

EXPECTED OUTPUT — GENERATED TEXT SEQUENCES (LSTM)

───────────────────────────────────────────────────────
  Seed : "deep learning"

  [temperature=0.6]
  deep learning models improve sequence learnin